# 02 Redis Key Schema

This notebook walks through the exact candidate-generation design used by the demo service.

The core idea is simple:
- user profiles live in `user:<id>` hashes,
- campaign metadata lives in `campaign:<id>` hashes,
- hard filters and segments are inverted into Redis Sets,
- the hot path performs a few `SINTER` calls and never scans all campaigns.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from redis import Redis


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.candidate import build_candidate_lookup_keys, build_indexes, filter_campaigns_for_user, generate_candidates_in_memory
from app.models import Campaign, UserProfile
from data.common import read_jsonl

DATASET_DIR = REPO_ROOT / 'data' / 'generated' / 'synthetic'
users = [UserProfile.model_validate(row) for row in read_jsonl(DATASET_DIR / 'users.jsonl')]
campaigns = [Campaign.model_validate(row) for row in read_jsonl(DATASET_DIR / 'campaigns.jsonl')]
indexes = build_indexes(campaigns)

sample_user = users[0]
sample_user

## What Gets Indexed

Each campaign contributes members to three index families:
- `idx:geo:<geo>`
- `idx:device:<device>`
- `idx:segment:<segment>`

This is enough to express the demo's hard filters while staying very fast and easy to inspect.

In [ ]:
index_rows = [
    {
        'key': key,
        'family': key.split(':')[1],
        'cardinality': len(values),
    }
    for key, values in indexes.items()
]
index_frame = pd.DataFrame(index_rows)
display(index_frame.groupby('family')['cardinality'].describe().round(2))
display(index_frame.sort_values('cardinality', ascending=False).head(12))

## Request-Time Lookup Plan

The service does not rely on a single query.
It tries a short ordered sequence of increasingly broad intersections:
1. `geo + device + strongest segments`
2. `geo + device + strongest single segment`
3. `geo + device`

This keeps recall reasonable without falling back to a full campaign scan.

In [ ]:
lookup_groups = build_candidate_lookup_keys(sample_user, strong_signal_count=2)
pd.DataFrame(
    {
        'stage': [f'lookup_{index + 1}' for index in range(len(lookup_groups))],
        'keys': lookup_groups,
        'redis_command': ['SINTER ' + ' '.join(group) for group in lookup_groups],
    }
)

## Candidate Pool Construction

The implementation deduplicates results across lookup stages and caps the pool size.
That cap is the hand-off point between retrieval and reranking.

In [ ]:
candidate_ids = generate_candidates_in_memory(sample_user, indexes, max_candidates=20, strong_signal_count=2)
candidate_campaigns = [campaign for campaign in campaigns if campaign.campaign_id in candidate_ids]
filtered = filter_campaigns_for_user(sample_user, candidate_campaigns)

pd.DataFrame(
    [
        {
            'campaign_id': campaign.campaign_id,
            'geo': campaign.geo,
            'device': campaign.device,
            'required_segments': campaign.required_segments,
        }
        for campaign in filtered[:10]
    ]
)

## Optional Live Redis Check

If the Docker stack is running, the cell below inspects the live Redis instance.
It shows the exact cardinalities and one real `SINTER` result for the sample user.

In [ ]:
redis_url = os.getenv('REDIS_URL', 'redis://localhost:6381/0')

try:
    client = Redis.from_url(redis_url, decode_responses=True)
    client.ping()
    live_rows = []
    for group in lookup_groups:
        live_rows.append(
            {
                'keys': group,
                'cardinalities': [client.scard(key) for key in group],
                'result_size': len(client.sinter(group)),
            }
        )
    pd.DataFrame(live_rows)
except Exception as exc:
    pd.Series({'redis_url': redis_url, 'status': f'not available: {exc}'})
